# 08d_provenance_filter — 출처·문헌으로 '실재 검증 천연물'만 정제 (신규)

**한 줄 요약:** NP-likeness를 통과한 59개 중, **COCONUT에 출처 생물(organisms) 또는 문헌(dois)이 기록된** 것만 남긴다 = 실제로 분리·보고된 천연물.
**왜:** NP-likeness는 "천연물처럼 생겼나"만 본다. 실재하는지(진짜 자연에서 분리됐는지)는 **출처·문헌 메타데이터**로 확인해야 한다. COCONUT엔 예측/파생 구조도 섞여 있음.
**큰 흐름:** ① 준비 → ② 후보+메타 로드 → ③ 출처/문헌 필터

> **📌 읽는 법**: 각 코드 셀은 [① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기].

### 준비 — 도구 불러오기

In [ ]:
import os
while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')  # data/ 폴더를 찾을 때까지 상위로 (하위 폴더에서 열어도 동작)
print('작업 폴더:', os.getcwd())
import pandas as pd

🔎 **코드 뜯어보기 (준비)**
- `pandas`만 필요(표 병합·필터).

### 셀 1 — 후보 + 메타데이터 병합
08c의 59개 후보에 COCONUT의 이름·출처·문헌·분류 열을 붙인다.

In [ ]:
# NP-likeness 통과 후보(08c) + COCONUT 메타데이터(출처·문헌·분류) 로드
CAND = "data/screen_repr_np_filtered.csv"          # 08c 결과(59개)
META = ["identifier", "name", "organisms", "dois",
        "np_classifier_pathway", "np_classifier_superclass", "np_classifier_class", "chemical_class"]
cand = pd.read_csv(CAND)
cc = pd.read_csv("data/coconut_csv-09-2026.csv", usecols=META)
df = cand.merge(cc, left_on="id", right_on="identifier", how="left")
print("NP통과 후보:", len(cand), "개 | 메타데이터 병합 완료")

🔎 **코드 뜯어보기 (셀 1)**
- `cand.merge(cc, left_on='id', right_on='identifier')` : 후보 ID로 COCONUT 메타를 붙임. `organisms`=분리된 생물, `dois`=논문, `np_classifier_*`=천연물 분류.

### 셀 2 — 출처/문헌 필터 & 저장
출처 생물 '또는' 문헌이 실제로 있는 후보만 남긴다(정체불명 제외).

In [ ]:
# '실재 검증' 필터: 출처 생물(organisms) '또는' 문헌(dois)이 실제로 있는 것만
def has(x):                                  # 값이 비었/NaN이 아니면 True
    return pd.notna(x) and str(x).strip() not in ("", "nan", "null")

df["has_org"] = df["organisms"].map(has)
df["has_doi"] = df["dois"].map(has)
verified = df[df.has_org | df.has_doi].copy()
verified = verified.sort_values(["np_score", "active_prob"], ascending=False).reset_index(drop=True)
verified.to_csv("data/screen_repr_np_provenance.csv", index=False)

print(f"실재 검증(출처 or 문헌 있음): {len(verified)} / {len(df)}개")
print(f"  - 출처 생물 있음: {df.has_org.sum()}개 | 문헌(DOI) 있음: {df.has_doi.sum()}개")
print(f"  - 둘 다 없어 제외(정체불명): {(~(df.has_org|df.has_doi)).sum()}개")
print("\n=== 실재 검증 천연물 후보 (전체) ===")
for _, x in verified.iterrows():
    nm = str(x["name"]) if has(x["name"]) else "(무명)"
    cls = str(x["np_classifier_class"]) if has(x["np_classifier_class"]) else "?"
    org = str(x["organisms"]) if has(x["organisms"]) else "-"
    print(f'  NP{x.np_score:.2f} prob{x.active_prob:.3f} | {x.id} | {cls}')
    print(f'      이름:{nm[:45]} | 출처:{org[:45]}')
print("\n저장: data/screen_repr_np_provenance.csv")

🔎 **코드 뜯어보기 (셀 2)**
- `has(x)` : 값이 비었거나 NaN/"nan"이면 False. 출처·문헌이 실제로 채워졌는지 판별.
- `df[df.has_org | df.has_doi]` : 출처 '또는' 문헌이 있는 행만(OR). 둘 다 없으면 실재 근거가 없어 제외.